# When Should Colombo Act? Dengue Outbreak Control Timing with a Calibrated SEIR Model

Dengue is a major public-health burden in Sri Lanka, and Colombo District reports more cases than any other district in most years. This notebook builds a seasonal SEIR (Susceptible–Exposed–Infectious–Recovered) model of dengue in Colombo, solves it with a hand-written fourth-order Runge–Kutta (RK4) method, calibrates it to weekly surveillance data with the Metropolis MCMC algorithm, and uses the fitted model to estimate how many cases a transmission-reducing intervention prevents when it starts earlier rather than later, with credible intervals from the posterior samples.

**Status:** the data, model and solver are complete; calibration, verification checks and the intervention experiments are added in later sections as the project progresses.

*Run with* **Runtime → Run all**. *Everything is downloaded from the public repository; no files or manual steps are needed.*

In [ ]:
# Setup: packages, seeds and data location.
# All of these are pre-installed on Google Colab; the install line makes local runs work too.
%pip install -q --disable-pip-version-check numpy scipy matplotlib pandas

import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt

NOTEBOOK_START = time.perf_counter()

SEED = 20261115                # master seed: every random generator below is derived from it
rng = np.random.default_rng(SEED)

REPO_RAW = "https://raw.githubusercontent.com/IT23226814/colombo-dengue-control-timing/main"
FIG_DIR = Path("figures")       # figures are also saved here (relative to the working directory)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
print(f"Seed: {SEED}")

## 1. Problem and question

Dengue cases in Colombo rise sharply in some years, usually after the monsoon rains. Health officials must decide **when** to start vector-control and clean-up campaigns: acting early costs resources before the outbreak is obvious, while acting late can mean thousands of extra cases.

**Question.** *If an intervention reduces dengue transmission in Colombo District by a given percentage, how many reported cases does it prevent when it starts earlier rather than later?*

The answer is given as cases prevented, with 95% credible intervals, for a range of start weeks and reduction levels, using a model calibrated to the 2026 weekly case series.

**Scope.** One district (Colombo), one season (2026, weeks 1–45), one aggregated intervention that lowers transmission by a fixed fraction from its start week. The mosquito population is not modelled explicitly in the core model.

## 2. Model, assumptions and equations

**State variables** (people): $S$ susceptible, $E$ exposed (infected, not yet infectious), $I$ infectious, $R$ recovered or immune, and $C$ the cumulative number of new symptomatic infections. Time $t$ is in **weeks**.

$$
\begin{aligned}
\frac{dS}{dt} &= -\beta(t)\,\frac{S\,I}{N}, &
\frac{dE}{dt} &= \beta(t)\,\frac{S\,I}{N} - \sigma E, &
\frac{dI}{dt} &= \sigma E - \gamma I, \\
\frac{dR}{dt} &= \gamma I, &
\frac{dC}{dt} &= \sigma E, &
N &= S + E + I + R .
\end{aligned}
$$

**Seasonal transmission with an intervention**

$$
\beta(t) = \beta_0\left[1 + a\cos\!\left(\frac{2\pi\,(t-\phi)}{52}\right)\right] u(t),
\qquad
u(t) = \begin{cases} 1 - r, & t_{\text{start}} \le t < t_{\text{end}} \\ 1, & \text{otherwise.} \end{cases}
$$

**Observation model.** New symptomatic infections in week $k$ are $C(k+1) - C(k)$; the expected number of **reported** cases is $\rho\,[C(k+1) - C(k)] + b$, where $\rho$ is the reporting rate and $b$ a constant background of reported cases per week.

**Basic reproduction number.** For a fully susceptible population, $R_0(t) = \beta(t)/\gamma$. If $R_0(t) < 1$ at all times, infections can only decline.

| Symbol | Meaning | Units | Value / source |
|---|---|---|---|
| $N$ | Colombo District population | people | 2,375,415 (Census of Population and Housing 2024) |
| $\beta_0$ | mean transmission rate | 1/week | estimated (MCMC) |
| $a$ | seasonal amplitude | – | estimated (MCMC), $0 \le a < 1$ |
| $\phi$ | week of peak transmission | weeks | estimated (MCMC) |
| $1/\sigma$ | intrinsic incubation period | weeks | 5.9 days = 0.843 weeks [3] |
| $1/\gamma$ | infectious (viraemic) period | weeks | 5 days = 0.714 weeks [4] |
| $\rho$ | reporting rate | – | estimated (MCMC) |
| $b$ | background reported cases | cases/week | estimated (MCMC) |
| $s_0$ | initial susceptible fraction | – | estimated (MCMC) |
| $r$, $t_{\text{start}}$ | intervention strength and start week | –, weeks | scenario inputs |

**Assumptions**
1. *Homogeneous mixing* within the district; no movement between districts.
2. *Host-only model:* transmission through *Aedes* mosquitoes, including the extrinsic incubation in the mosquito (about 8–12 days [4]), is absorbed into $\beta(t)$. This is the main structural simplification; a host–vector extension is a stretch goal.
3. *Seasonality* is a single annual cosine in $\beta$, standing in for rainfall- and temperature-driven mosquito abundance.
4. *Immunity:* only a fraction $s_0$ of the population is susceptible at the start of the year (past exposure to dengue); recovery gives immunity for the rest of the season. Serotypes are not distinguished.
5. *Reporting:* reported cases are a fixed fraction $\rho$ of the season's symptomatic infections plus a constant background $b$ of cases that the single-season dynamics do not produce (endemic transmission in pockets of the district, imported infections, and suspected cases that are not dengue); reporting delays are ignored (the data already use revised weekly counts).
6. *Intervention:* any combination of measures is summarised as a fractional cut $r$ in transmission from week $t_{\text{start}}$.

**Concepts applied:** compartmental epidemic models (SEIR); numerical solution of ODEs with the classical fourth-order Runge–Kutta method and its error order; Bayesian calibration with the Metropolis MCMC algorithm; Monte Carlo propagation of parameter uncertainty into credible intervals.

## 3. Implementation

### 3.1 Data

Weekly suspected dengue cases for Colombo District, taken from the National Dengue Control Unit's *Weekly Dengue Update* reports [1], and the district population from the 2024 census [2]. How the series was extracted and checked is documented in [`data/README.md`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/data/README.md).

*Provenance: data-loading code written with an AI coding agent.*

In [ ]:
cases = pd.read_csv(f"{REPO_RAW}/data/colombo_weekly_cases.csv", parse_dates=["week_start", "week_end"])
population = pd.read_csv(f"{REPO_RAW}/data/colombo_population.csv")
N_COLOMBO = int(population.loc[0, "population"])          # people

obs = cases[cases.year == 2026].sort_values("week").reset_index(drop=True)
WEEKS = len(obs)                                           # number of observed weeks
y_obs = obs["cases"].to_numpy()                            # reported cases per week

# Sanity checks on the input data
assert N_COLOMBO == 2_375_415
assert obs["week"].tolist() == list(range(1, WEEKS + 1)), "2026 weeks must be consecutive from week 1"
assert (y_obs >= 0).all() and not obs["cases"].isna().any()

print(f"Population N = {N_COLOMBO:,}")
print(f"2026 data: weeks 1–{WEEKS} ({obs.week_start.iloc[0].date()} to {obs.week_end.iloc[-1].date()}), "
      f"{y_obs.sum():,} reported cases, peak {y_obs.max():,} in week {obs.week[y_obs.argmax()]}")

fig, ax = plt.subplots()
for year, colour in [(2025, "0.55"), (2026, "C3")]:
    d = cases[cases.year == year].set_index("week")["cases"].reindex(range(1, 53))  # gaps stay empty
    ax.plot(d.index, d.values, "o-", ms=3, color=colour, label=str(year))
ax.set(xlabel="Week of year", ylabel="Reported cases per week", title="Weekly dengue cases, Colombo District", xlim=(0, 53))
ax.legend(frameon=False)
plt.show()

### 3.2 SEIR model and RK4 solver

The model is written from scratch: the right-hand side `rhs`, a classical RK4 step `rk4_step` (four stages, local error $O(h^5)$, global error $O(h^4)$), and `simulate`, which advances the state with a fixed step of $h = 1/7$ week (one day). The cumulative compartment $C$ gives exact weekly incidence as a difference of two RK4 states. Every parameter may be a NumPy array, so many parameter sets (e.g. posterior samples) are simulated in one call.

The same code is kept in [`src/seir.py`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/src/seir.py), where an automated test suite also runs against it; a test checks that this cell and the file stay identical.

*Provenance: written with an AI coding agent; see the project's AI-use record.*

In [ ]:
"""Seasonal SEIR model of dengue transmission, solved with a hand-written RK4.

Units used throughout
---------------------
time            weeks (t = 0 is the start of the first simulated week)
rates           1/week (beta0, sigma, gamma)
compartments    people (S, E, I, R and N)
C               cumulative new symptomatic infections (people), dC/dt = sigma*E

Model
-----
    dS/dt = -beta(t) S I / N
    dE/dt =  beta(t) S I / N - sigma E
    dI/dt =  sigma E - gamma I
    dR/dt =  gamma I
    dC/dt =  sigma E                      (counts every E -> I transition)

    beta(t) = beta0 * [1 + amp * cos(2 pi (t - phase) / period)] * u(t)
    u(t)    = 1 - reduction  while start <= t < end  (intervention), else 1

Weekly new infections in week k (from t = k to k + 1) are C(k+1) - C(k), and
the expected *reported* cases are rho times that.

This is a host-only model: the mosquito stage is not modelled explicitly, so
transmission through mosquitoes is absorbed into beta(t).

Every parameter may be a float or a NumPy array. Arrays are broadcast against
each other, so one call can run many parameter sets (e.g. MCMC samples) at once.
"""

from __future__ import annotations

from dataclasses import dataclass, fields, replace

import numpy as np

DAYS_PER_WEEK = 7.0

# Literature defaults for the natural history (converted to 1/week):
#   mean intrinsic incubation period 5.9 days  [Chan & Johansson 2012, PLoS ONE 7(11):e50972,
#                                               doi:10.1371/journal.pone.0050972]
#   human viraemic (infectious) period ~5 days [WHO dengue fact sheet: "about 4-5 days"]
SIGMA_DEFAULT = DAYS_PER_WEEK / 5.9
GAMMA_DEFAULT = DAYS_PER_WEEK / 5.0


@dataclass(frozen=True)
class Params:
    """Model parameters. Floats or broadcastable NumPy arrays."""

    N: float                      # population (people)
    beta0: float                  # mean transmission rate (1/week)
    amp: float = 0.0              # seasonal amplitude, 0 <= amp < 1 (dimensionless)
    phase: float = 0.0            # week of peak transmission (weeks)
    period: float = 52.0          # length of the seasonal cycle (weeks)
    sigma: float = SIGMA_DEFAULT  # 1 / incubation period (1/week)
    gamma: float = GAMMA_DEFAULT  # 1 / infectious period (1/week)
    rho: float = 1.0              # reporting rate: reported / symptomatic infections
    s0: float = 1.0               # initial susceptible fraction of N
    e0: float = 0.0               # initial exposed (people)
    i0: float = 1.0               # initial infectious (people)
    start: float = np.inf         # intervention start (weeks)
    end: float = np.inf           # intervention end (weeks)
    reduction: float = 0.0        # fractional cut in beta during the intervention

    def with_(self, **changes) -> "Params":
        """Return a copy with some parameters changed."""
        return replace(self, **changes)

    def arrays(self) -> dict[str, np.ndarray]:
        """All parameters as NumPy arrays broadcast to one common shape."""
        names = [f.name for f in fields(self)]
        vals = np.broadcast_arrays(*[np.asarray(getattr(self, n), dtype=float) for n in names])
        return dict(zip(names, vals))


def control(t: float, p: dict[str, np.ndarray]) -> np.ndarray:
    """Intervention multiplier u(t): 1 - reduction while start <= t < end, else 1."""
    active = (t >= p["start"]) & (t < p["end"])
    return np.where(active, 1.0 - p["reduction"], 1.0)


def beta(t: float, p: dict[str, np.ndarray], u: np.ndarray | None = None) -> np.ndarray:
    """Transmission rate at time t (1/week). u is the intervention multiplier;
    if None it is evaluated at t."""
    seasonal = p["beta0"] * (1.0 + p["amp"] * np.cos(2.0 * np.pi * (t - p["phase"]) / p["period"]))
    return seasonal * (control(t, p) if u is None else u)


def basic_reproduction_number(t: float, params: Params) -> np.ndarray:
    """Instantaneous R0(t) = beta(t) / gamma for a fully susceptible population."""
    p = params.arrays()
    return beta(t, p) / p["gamma"]


def rhs(t: float, y: np.ndarray, p: dict[str, np.ndarray], u: np.ndarray | None = None) -> np.ndarray:
    """Right-hand side f(t, y); y has shape (5, ...) = (S, E, I, R, C)."""
    S, E, I, R, C = y
    infection = beta(t, p, u) * S * I / p["N"]
    incidence = p["sigma"] * E
    recovery = p["gamma"] * I
    return np.stack([-infection, infection - incidence, incidence - recovery, recovery, incidence])


def rk4_step(f, t: float, y: np.ndarray, h: float, *args) -> np.ndarray:
    """One classical 4th-order Runge-Kutta step of size h for y' = f(t, y, *args)."""
    k1 = f(t, y, *args)
    k2 = f(t + h / 2, y + h / 2 * k1, *args)
    k3 = f(t + h / 2, y + h / 2 * k2, *args)
    k4 = f(t + h, y + h * k3, *args)
    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def initial_state(p: dict[str, np.ndarray]) -> np.ndarray:
    """(S, E, I, R, C) at t = 0. People not susceptible, exposed or infectious start in R."""
    S = p["s0"] * p["N"] - p["e0"] - p["i0"]
    R = p["N"] - S - p["e0"] - p["i0"]
    if np.any(S < 0):
        raise ValueError("s0 * N must be at least e0 + i0")
    return np.stack([S, p["e0"], p["i0"], R, np.zeros_like(S)])


def simulate(params: Params, weeks: int, steps_per_week: int = 7) -> dict[str, np.ndarray]:
    """Integrate the model for `weeks` weeks with fixed RK4 steps of 1/steps_per_week weeks.

    Returns a dict with
      t                 (weeks*steps_per_week + 1,) time grid in weeks
      y                 (len(t), 5, ...) states S, E, I, R, C on the grid
      weekly_infections (weeks, ...) new symptomatic infections in each week
      weekly_reported   (weeks, ...) expected reported cases = rho * weekly_infections

    The intervention multiplier u is held constant over each step, at its value
    at the step midpoint, so the right-hand side is smooth within every step.
    Start/end times on the step grid (e.g. whole weeks) are therefore handled
    exactly and keep full 4th-order accuracy.
    """
    p = params.arrays()
    h = 1.0 / steps_per_week
    n = weeks * steps_per_week
    y = initial_state(p)
    out = np.empty((n + 1,) + y.shape)
    out[0] = y
    for k in range(n):
        t = k * h
        y = rk4_step(rhs, t, y, h, p, control(t + h / 2, p))
        out[k + 1] = y
    C_weekly = out[::steps_per_week, 4]
    infections = np.diff(C_weekly, axis=0)
    return {
        "t": np.arange(n + 1) * h,
        "y": out,
        "weekly_infections": infections,
        "weekly_reported": p["rho"] * infections,
    }

**Example run (illustrative parameters, not fitted).** The values below only show that the model produces a seasonal outbreak and that an intervention lowers it; the calibrated values come from Section 3.3.

In [ ]:
example = Params(N=N_COLOMBO, beta0=2.8, amp=0.5, phase=22.0, rho=0.1, s0=0.5, e0=1500.0, i0=1500.0)
weeks = np.arange(1, WEEKS + 1)

baseline = simulate(example, WEEKS)
intervention = simulate(example.with_(start=18, reduction=0.25), WEEKS)

fig, ax = plt.subplots()
ax.plot(weeks, baseline["weekly_reported"], label="no intervention")
ax.plot(weeks, intervention["weekly_reported"], label="25% cut in transmission from week 18")
ax.axvline(18, color="0.5", ls=":")
ax.set(xlabel="Week of 2026", ylabel="Expected reported cases per week", title="Illustrative model run (not fitted)")
ax.legend(frameon=False)
plt.show()

prevented = baseline["weekly_reported"].sum() - intervention["weekly_reported"].sum()
print(f"Illustrative only: {prevented:,.0f} reported cases prevented over weeks 1–{WEEKS}")

### 3.3 Calibration with Metropolis MCMC

**Likelihood.** The observed weekly counts $y_k$ are modelled as negative binomial around the model's expected reported cases $\mu_k(\theta) = \rho\,[C(k+1) - C(k)] + b$ (Section 2):

$$
y_k \sim \text{NegBin}(\text{mean } \mu_k,\ \text{dispersion } \kappa), \qquad \operatorname{Var}(y_k) = \mu_k + \mu_k^2/\kappa ,
$$
$$
\log p(y \mid \theta) = \sum_{k}\Big[\log\Gamma(y_k+\kappa) - \log\Gamma(\kappa) - \log y_k! + \kappa\log\tfrac{\kappa}{\kappa+\mu_k} + y_k\log\tfrac{\mu_k}{\kappa+\mu_k}\Big].
$$

A Poisson likelihood ($\kappa \to \infty$) would force the variance to equal the mean, but surveillance counts vary much more than that from week to week (reporting effort, holidays, batch uploads, and the model's own simplifications), so a Poisson fit would be over-confident. The dispersion $\kappa$ is estimated with the other parameters, so the data decide how much extra noise there is.

**Priors** are stated on the natural scale of each parameter. Bounded parameters are sampled on a logit scale and positive ones on a log scale, $z = \operatorname{logit}\!\big(\tfrac{\theta - \ell}{u - \ell}\big)$ or $z = \log\theta$, and the log-Jacobian $\log|d\theta/dz|$ is added to the log-density so the prior on $\theta$ is unchanged. Sampling on $z$ removes the hard boundaries and makes the posterior closer to Gaussian.

**Metropolis algorithm.** For each chain, with current state $z$:

1. propose $z' = z + L\,\varepsilon$ with $\varepsilon \sim \mathcal N(0, I)$ (a symmetric random walk);
2. accept with probability $\alpha = \min\{1, \exp[\log p(z' \mid y) - \log p(z \mid y)]\}$, otherwise stay at $z$.

Because the proposal is symmetric, the proposal densities cancel and this kernel leaves the posterior invariant (detailed balance). **Tuning:** during burn-in the proposal covariance $LL^\top$ is set to $\frac{2.38^2}{d}\hat\Sigma$ from the chains' recent samples, and its overall size is nudged towards an acceptance rate of about 0.25. After burn-in the proposal is **frozen**, so the samples that are kept come from a fixed Markov chain; adapting after that point would break the Markov property.

**Convergence diagnostics.** Several chains start from widely spread points. The split-$\hat R$ statistic compares the variance between and within (halves of) chains; values near 1 mean the chains have forgotten their starts and agree. The effective sample size (ESS) accounts for autocorrelation and sets the Monte Carlo error of posterior means, $\text{sd}/\sqrt{\text{ESS}}$.

The sampler, priors, likelihood and diagnostics below are written from scratch (SciPy is used only for $\log\Gamma$). The same code is kept in [`src/mcmc.py`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/src/mcmc.py) with its own automated tests.

*Provenance: written with an AI coding agent.*

In [ ]:
"""Random-walk Metropolis MCMC, priors on transformed scales, convergence
diagnostics, and the posterior for the seasonal SEIR model.

Everything is written from scratch with NumPy; SciPy is used only for the
log-gamma function. Several chains run side by side as rows of one array, so
each iteration evaluates the log-posterior of all chains in one vectorised call
(the SEIR model broadcasts over parameter arrays).

Sampling space
--------------
Each parameter theta is sampled on an unconstrained scale z:
    bounded (lo, hi)  theta = lo + (hi - lo) * logistic(z)
    positive          theta = exp(z)
    real              theta = z
The density on z includes the log-Jacobian |d theta / d z|, so the prior on
theta is exactly the one stated.

Metropolis algorithm (per chain)
--------------------------------
    propose z' = z + L e,  e ~ N(0, I)          (symmetric proposal)
    accept with probability min(1, exp(log p(z') - log p(z)))
During burn-in the proposal covariance L L^T is adapted from the chains'
recent samples (scaled by 2.38^2 / d) and its overall scale is tuned towards
an acceptance rate of about 0.25. After burn-in the proposal is frozen, so
the kept samples come from a fixed Metropolis kernel with the posterior as its
stationary distribution.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
from scipy.special import gammaln

# ---------------------------------------------------------------------------
# Priors and transforms
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class Prior:
    """Prior for one parameter.

    kind: "uniform" (on lo..hi), "normal" (mu, sd, truncated to lo..hi),
          "lognormal" (mu, sd of log theta; positive), "beta" (a, b on lo..hi).
    The sampling transform follows from the support: bounded -> logit,
    positive -> log, otherwise identity.
    """

    kind: str
    lo: float = -np.inf
    hi: float = np.inf
    mu: float = 0.0
    sd: float = 1.0
    a: float = 1.0
    b: float = 1.0

    @property
    def transform(self) -> str:
        if np.isfinite(self.lo) and np.isfinite(self.hi):
            return "logit"
        if self.lo == 0.0 and not np.isfinite(self.hi):
            return "log"
        return "identity"

    def log_density(self, theta: np.ndarray) -> np.ndarray:
        """log p(theta) up to an additive constant; -inf outside the support."""
        theta = np.asarray(theta, dtype=float)
        inside = (theta > self.lo) & (theta < self.hi)
        with np.errstate(divide="ignore", invalid="ignore"):
            if self.kind == "uniform":
                lp = np.zeros_like(theta)
            elif self.kind == "normal":
                lp = -0.5 * ((theta - self.mu) / self.sd) ** 2
            elif self.kind == "lognormal":
                lp = -np.log(theta) - 0.5 * ((np.log(theta) - self.mu) / self.sd) ** 2
            elif self.kind == "beta":
                x = (theta - self.lo) / (self.hi - self.lo)
                lp = (self.a - 1) * np.log(x) + (self.b - 1) * np.log1p(-x)
            else:
                raise ValueError(f"unknown prior kind {self.kind!r}")
        return np.where(inside, lp, -np.inf)

    def to_theta(self, z: np.ndarray) -> np.ndarray:
        if self.transform == "logit":
            return self.lo + (self.hi - self.lo) / (1.0 + np.exp(-z))
        if self.transform == "log":
            return np.exp(z)
        return z

    def to_z(self, theta: np.ndarray) -> np.ndarray:
        theta = np.asarray(theta, dtype=float)
        if self.transform == "logit":
            x = (theta - self.lo) / (self.hi - self.lo)
            return np.log(x) - np.log1p(-x)
        if self.transform == "log":
            return np.log(theta)
        return theta

    def log_jacobian(self, z: np.ndarray) -> np.ndarray:
        """log |d theta / d z|."""
        if self.transform == "logit":
            # d/dz logistic(z) = s (1 - s); computed stably with logaddexp
            return np.log(self.hi - self.lo) - np.logaddexp(0.0, z) - np.logaddexp(0.0, -z)
        if self.transform == "log":
            return z
        return np.zeros_like(z)


@dataclass(frozen=True)
class Model:
    """A named set of priors and a log-likelihood log p(data | theta).

    log_likelihood receives a dict {name: array of shape (n_chains,)} and
    returns an array of shape (n_chains,).
    """

    priors: dict[str, Prior]
    log_likelihood: object = field(repr=False)

    @property
    def names(self) -> list[str]:
        return list(self.priors)

    def to_theta(self, z: np.ndarray) -> dict[str, np.ndarray]:
        return {n: p.to_theta(z[..., j]) for j, (n, p) in enumerate(self.priors.items())}

    def to_z(self, theta: dict[str, np.ndarray]) -> np.ndarray:
        return np.stack([p.to_z(theta[n]) for n, p in self.priors.items()], axis=-1)

    def log_posterior(self, z: np.ndarray) -> np.ndarray:
        """log p(z | data) up to a constant, for z of shape (n_chains, d)."""
        theta = self.to_theta(z)
        lp = np.zeros(z.shape[0])
        for j, (n, p) in enumerate(self.priors.items()):
            lp = lp + p.log_density(theta[n]) + p.log_jacobian(z[:, j])
        ok = np.isfinite(lp)
        if ok.any():
            ll = np.full(z.shape[0], -np.inf)
            sub = {n: v[ok] for n, v in theta.items()}
            ll[ok] = self.log_likelihood(sub)
            lp = np.where(ok, lp + ll, -np.inf)
        return np.where(np.isnan(lp), -np.inf, lp)


# ---------------------------------------------------------------------------
# Likelihood for counts
# ---------------------------------------------------------------------------


def negbin_logpmf(y: np.ndarray, mean: np.ndarray, k: np.ndarray) -> np.ndarray:
    """log P(Y = y) for a negative binomial with the given mean and dispersion k.

    Var(Y) = mean + mean^2 / k, so k -> infinity recovers the Poisson.
    """
    mean = np.maximum(mean, 1e-12)
    return (gammaln(y + k) - gammaln(k) - gammaln(y + 1)
            + k * (np.log(k) - np.log(k + mean)) + y * (np.log(mean) - np.log(k + mean)))


# ---------------------------------------------------------------------------
# Metropolis sampler
# ---------------------------------------------------------------------------


@dataclass
class MCMCResult:
    names: list[str]
    z: np.ndarray            # (n_keep, n_chains, d) kept samples on the sampling scale
    theta: dict              # {name: (n_keep, n_chains)} kept samples on the natural scale
    log_post: np.ndarray     # (n_keep, n_chains)
    acceptance: np.ndarray   # (n_chains,) acceptance rate after burn-in
    proposal_cov: np.ndarray # (d, d) frozen proposal covariance used after burn-in
    burn_in_trace: dict      # {name: (n_burn, n_chains)} for trace plots

    def flat(self, name: str) -> np.ndarray:
        """All kept samples of one parameter, chains pooled."""
        return self.theta[name].reshape(-1)


def metropolis(model: Model, z0: np.ndarray, n_burn: int, n_keep: int, rng: np.random.Generator,
               init_scale: float = 0.1, adapt_every: int = 100, target_accept: float = 0.25,
               thin: int = 1) -> MCMCResult:
    """Run len(z0) random-walk Metropolis chains in parallel.

    z0: (n_chains, d) starting points on the sampling scale (use different,
        over-dispersed starts so convergence can be checked).
    n_burn: iterations with proposal adaptation (discarded).
    n_keep: iterations kept after burn-in (before thinning).
    """
    z = np.array(z0, dtype=float)
    n_chains, d = z.shape
    lp = model.log_posterior(z)
    if not np.all(np.isfinite(lp)):
        raise ValueError("every chain must start where the posterior is finite")

    cov = np.eye(d) * init_scale**2          # proposal covariance before adaptation
    log_scale = 0.0                          # multiplies the covariance by exp(2 * log_scale)
    burn = np.empty((n_burn, n_chains, d))
    window_accepts = 0

    def step(z, lp, chol):
        proposal = z + rng.standard_normal((n_chains, d)) @ chol.T
        lp_new = model.log_posterior(proposal)
        accept = np.log(rng.random(n_chains)) < lp_new - lp
        z = np.where(accept[:, None], proposal, z)
        lp = np.where(accept, lp_new, lp)
        return z, lp, accept

    chol = np.linalg.cholesky(cov)
    for i in range(n_burn):
        z, lp, acc = step(z, lp, np.exp(log_scale) * chol)
        burn[i] = z
        window_accepts += acc.sum()
        if (i + 1) % adapt_every == 0:
            rate = window_accepts / (adapt_every * n_chains)
            log_scale += (rate - target_accept) * 2.0 / np.sqrt((i + 1) / adapt_every)
            window_accepts = 0
            # empirical covariance of the second half of burn-in so far, all chains pooled
            recent = burn[(i + 1) // 2: i + 1].reshape(-1, d)
            if recent.shape[0] > 10 * d:
                emp = np.cov(recent, rowvar=False) + 1e-8 * np.eye(d)
                cov = 2.38**2 / d * emp
                chol = np.linalg.cholesky(cov)

    final_chol = np.exp(log_scale) * chol    # frozen from here on
    keep = np.empty((n_keep // thin, n_chains, d))
    keep_lp = np.empty((n_keep // thin, n_chains))
    accepted = np.zeros(n_chains)
    for i in range(n_keep):
        z, lp, acc = step(z, lp, final_chol)
        accepted += acc
        if (i + 1) % thin == 0:
            keep[(i + 1) // thin - 1] = z
            keep_lp[(i + 1) // thin - 1] = lp

    theta = model.to_theta(keep)
    return MCMCResult(
        names=model.names, z=keep, theta=theta, log_post=keep_lp,
        acceptance=accepted / n_keep, proposal_cov=final_chol @ final_chol.T,
        burn_in_trace=model.to_theta(burn),
    )


# ---------------------------------------------------------------------------
# Convergence diagnostics
# ---------------------------------------------------------------------------


def split_rhat(x: np.ndarray) -> float:
    """Split-R-hat (Gelman et al.) for samples x of shape (n_iter, n_chains).

    Each chain is split in half; values close to 1 (below about 1.01-1.05)
    indicate that the chains agree with each other and with themselves.
    """
    n = x.shape[0] // 2
    halves = np.concatenate([x[:n], x[n:2 * n]], axis=1)       # (n, 2 * n_chains)
    chain_means = halves.mean(axis=0)
    W = halves.var(axis=0, ddof=1).mean()                       # within-chain variance
    B = n * chain_means.var(ddof=1)                             # between-chain variance
    var_plus = (n - 1) / n * W + B / n
    return float(np.sqrt(var_plus / W))


def effective_sample_size(x: np.ndarray) -> float:
    """Effective sample size for x of shape (n_iter, n_chains), using the
    multi-chain autocorrelation and Geyer's initial positive sequence."""
    n, m = x.shape
    centred = x - x.mean(axis=0)
    f = np.fft.rfft(centred, n=2 * n, axis=0)
    acov = np.fft.irfft(f * np.conj(f), axis=0)[:n] / n          # (n, m) autocovariance per chain
    W = x.var(axis=0, ddof=1).mean()
    var_plus = (n - 1) / n * W + x.mean(axis=0).var(ddof=1) if m > 1 else W
    rho = 1.0 - (W - acov.mean(axis=1)) / var_plus
    rho[0] = 1.0
    tau = -1.0
    for t in range(0, n - 1, 2):                                 # sum pairs while positive
        pair = rho[t] + rho[t + 1]
        if pair < 0:
            break
        tau += 2.0 * pair
    return float(n * m / max(tau, 1e-12))


def summarise(result: MCMCResult) -> list[dict]:
    """Posterior mean, 95% credible interval, split-R-hat and ESS for each parameter."""
    rows = []
    for name in result.names:
        x = result.theta[name]
        lo, hi = np.percentile(x, [2.5, 97.5])
        rows.append(dict(parameter=name, mean=x.mean(), sd=x.std(ddof=1), ci_2_5=lo, ci_97_5=hi,
                         rhat=split_rhat(x), ess=effective_sample_size(x)))
    return rows


# ---------------------------------------------------------------------------
# Posterior for the seasonal SEIR model
# ---------------------------------------------------------------------------


def seir_model(y_obs: np.ndarray, priors: dict[str, Prior], fixed: dict, simulate, Params,
               steps_per_week: int = 7) -> Model:
    """Negative-binomial likelihood of weekly reported cases under the SEIR model.

    Expected reported cases in week k are  rho * (new infections in week k) + bg,
    where bg is an optional constant background of reported cases per week.

    priors: priors for any of beta0, amp, phase, rho, s0, i0 (initial
            infectious people), k (negative-binomial dispersion) and bg
            (background reported cases per week; omit it to fix bg = 0).
    fixed:  values for everything not estimated: Params fields (must include
            N) and, optionally, k and bg. Without k (estimated or fixed) the
            likelihood is Poisson.
    Initial exposed people are set to e0 = i0 * gamma / sigma, the ratio at
    which E and I are in balance when incidence is roughly steady.
    simulate, Params: the model's functions, passed in so this module does
    not depend on how the model is imported.
    """
    y_obs = np.asarray(y_obs, dtype=float)
    weeks = len(y_obs)
    observation_fields = {"k", "bg"}
    fixed_model = {n: v for n, v in fixed.items() if n not in observation_fields}
    fixed_obs = {n: v for n, v in fixed.items() if n in observation_fields}

    def log_likelihood(theta: dict) -> np.ndarray:
        kwargs = dict(fixed_model)
        kwargs.update({n: v for n, v in theta.items() if n not in observation_fields})
        p = Params(**kwargs)
        arr = p.arrays()
        n_chains = len(next(iter(theta.values())))
        i0 = np.broadcast_to(arr["i0"], (n_chains,)).copy()
        ratio = np.broadcast_to(arr["gamma"] / arr["sigma"], (n_chains,))
        # Parameter sets with more initial infected than susceptible people are impossible:
        # give them a harmless value for the run and a log-likelihood of -inf afterwards.
        bad = np.broadcast_to(arr["s0"] * arr["N"], (n_chains,)) < i0 * (1.0 + ratio)
        i0[bad] = 1.0
        p = p.with_(i0=i0, e0=i0 * ratio)
        mean = simulate(p, weeks, steps_per_week)["weekly_reported"]
        mean = np.broadcast_to(mean.reshape(weeks, -1), (weeks, n_chains))  # (weeks, n_chains)
        obs = {**fixed_obs, **{n: v for n, v in theta.items() if n in observation_fields}}
        if "bg" in obs:
            mean = mean + obs["bg"]
        if "k" in obs:
            ll = negbin_logpmf(y_obs[:, None], mean, obs["k"]).sum(axis=0)
        else:                                                          # Poisson likelihood
            m = np.maximum(mean, 1e-12)
            ll = (y_obs[:, None] * np.log(m) - m - gammaln(y_obs[:, None] + 1)).sum(axis=0)
        return np.where(bad, -np.inf, ll)

    return Model(priors=priors, log_likelihood=log_likelihood)

**Sampler check on a target with a known answer.** Before using the sampler on the epidemic model, it is run on a correlated two-dimensional Gaussian whose mean and covariance are known exactly. Six chains start far apart; after burn-in the sample mean must be within 4 Monte Carlo standard errors of the true mean, the covariance must match, split-$\hat R$ must be below 1.02, and the acceptance rate must be in the tuned range.

In [ ]:
# Sampler check: correlated 2-D Gaussian with known mean and covariance
true_mean = np.array([1.0, -2.0])
true_cov = np.array([[1.0, 0.8], [0.8, 2.0]])
true_prec = np.linalg.inv(true_cov)


def gaussian_loglik(theta):
    d = np.stack([theta["x"], theta["y"]], axis=-1) - true_mean
    return -0.5 * np.einsum("ni,ij,nj->n", d, true_prec, d)


gauss = Model(priors={"x": Prior("uniform", lo=-50, hi=50), "y": Prior("uniform", lo=-50, hi=50)},
              log_likelihood=gaussian_loglik)
gauss_rng = np.random.default_rng([SEED, 33])
starts = gauss.to_z({"x": gauss_rng.uniform(-8, 8, 6), "y": gauss_rng.uniform(-8, 8, 6)})
gres = metropolis(gauss, starts, n_burn=3000, n_keep=6000, rng=gauss_rng)

samples = np.stack([gres.flat("x"), gres.flat("y")], axis=1)
for j, name in enumerate(["x", "y"]):
    ess = effective_sample_size(gres.theta[name])
    mc_se = np.sqrt(true_cov[j, j] / ess)
    rhat = split_rhat(gres.theta[name])
    print(f"{name}: mean {samples[:, j].mean():+.3f} (true {true_mean[j]:+.1f}, MC s.e. {mc_se:.3f}), "
          f"R-hat {rhat:.4f}, ESS {ess:,.0f}")
    assert abs(samples[:, j].mean() - true_mean[j]) < 4 * mc_se
    assert rhat < 1.02
print("sample covariance:\n", np.round(np.cov(samples, rowvar=False), 3))
print("acceptance rate per chain:", np.round(gres.acceptance, 3))
assert np.allclose(np.cov(samples, rowvar=False), true_cov, rtol=0.15, atol=0.1)
assert np.all((gres.acceptance > 0.15) & (gres.acceptance < 0.45))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
n_burn_plot = gres.burn_in_trace["x"].shape[0]
for c in range(6):
    trace = np.concatenate([gres.burn_in_trace["x"][:, c], gres.theta["x"][:, c]])
    axes[0].plot(trace, lw=0.4)
axes[0].axvline(n_burn_plot, color="k", ls="--", lw=0.8)
axes[0].set(xlabel="Iteration", ylabel="x", title="Trace of x for 6 chains (dashed: end of burn-in)")
axes[1].plot(samples[::10, 0], samples[::10, 1], ".", ms=1.5, alpha=0.4)
axes[1].set(xlabel="x", ylabel="y", title="Kept samples (every 10th)")
plt.tight_layout()
plt.show()
print("PASS: the sampler recovers the known mean and covariance, and the chains agree")

**Posterior for the SEIR model.** The parameters estimated from the data and their priors:

| Parameter | Meaning | Prior | Reasoning |
|---|---|---|---|
| $\beta_0$ | mean transmission rate (1/week) | Uniform(0.5, 10) | mean $R_0=\beta_0/\gamma$ between about 0.4 and 7 |
| $a$ | seasonal amplitude | Uniform(0, 0.95) | from no seasonality to near-complete seasonal swing; must be below 1 to keep $\beta>0$ |
| $\phi$ | week of peak transmission | Uniform(10, 40) | transmission peaks before the case peak; the data peak in week 27 |
| $\rho$ | reporting rate | Uniform(0.005, 1) | most dengue infections are mild or unreported; no stronger assumption |
| $s_0$ | initial susceptible fraction | Uniform(0.01, 1) | unknown share of Colombo residents without immunity |
| $I_0$ | infectious people at week 0 | LogNormal on $[0,\infty)$, median 1,000, log-sd 1.5 | wide; sets the size of the early-year epidemic; $E_0 = I_0\gamma/\sigma$ |
| $\kappa$ | negative-binomial dispersion | LogNormal, median 20, log-sd 1.5 | from strong to mild overdispersion |
| $b$ | background reported cases per week | Uniform(0, 1000) | steady endemic/imported cases not produced by the season's outbreak dynamics |

The incubation and infectious periods stay fixed at their literature values (Section 2). The posterior is built by `seir_model(...)` from the code above; it is run on synthetic data in Section 4.4 and on the 2026 Colombo data in Section 3.4.

In [ ]:
# Priors for the SEIR calibration (natural scale)
SEIR_PRIORS = {
    "beta0": Prior("uniform", lo=0.5, hi=10.0),                  # 1/week
    "amp":   Prior("uniform", lo=0.0, hi=0.95),
    "phase": Prior("uniform", lo=10.0, hi=40.0),                 # weeks
    "rho":   Prior("uniform", lo=0.005, hi=1.0),
    "s0":    Prior("uniform", lo=0.01, hi=1.0),
    "i0":    Prior("lognormal", lo=0.0, mu=np.log(1000.0), sd=1.5),  # people
    "k":     Prior("lognormal", lo=0.0, mu=np.log(20.0), sd=1.5),
    "bg":    Prior("uniform", lo=0.0, hi=1000.0),                # reported cases per week
}
FIXED = dict(N=N_COLOMBO)                                        # sigma, gamma: literature defaults


def colombo_posterior(y, steps_per_week=7):
    # Log-posterior of the SEIR parameters given weekly reported cases y
    return seir_model(y, SEIR_PRIORS, FIXED, simulate, Params, steps_per_week)


# Quick check: the log-posterior is finite at the illustrative parameters and works for many chains at once
post = colombo_posterior(y_obs)
z_try = post.to_z({"beta0": np.array([2.8, 2.0]), "amp": np.array([0.5, 0.3]), "phase": np.array([22.0, 25.0]),
                   "rho": np.array([0.1, 0.2]), "s0": np.array([0.5, 0.4]), "i0": np.array([1500.0, 800.0]),
                   "k": np.array([20.0, 10.0]), "bg": np.array([1.0, 250.0])})
print("log-posterior at two example points:", np.round(post.log_posterior(z_try), 1))
assert np.all(np.isfinite(post.log_posterior(z_try)))

### 3.4 Fit to the 2026 Colombo data

*To be added: the sampler run on the 2026 weekly series, trace plots, posterior summaries with $\hat R$ and ESS, and a posterior predictive check.*

## 4. Verification

Each check below ends in an `assert`, so *Run all* stops with an error if the simulator is wrong. The checks test the solver against facts that must hold whatever the parameter values are, so they do not depend on the calibration.

### 4.1 Conservation of population

The four compartment equations sum to zero, $\frac{d}{dt}(S+E+I+R) = 0$, so the total population must stay at $N$ for all time. RK4 is a linear combination of right-hand-side evaluations, so it preserves this linear invariant exactly apart from floating-point rounding. A second consistency check follows from $C' = \sigma E$ and $S' + E' = -\sigma E$: the cumulative count of new infections must equal the number of people who have left $S$ and $E$, $C(t) = [S(0)-S(t)] - [E(t)-E(0)]$.

The check runs 200 random parameter sets at once (wide ranges of $\beta_0$, $a$, $\phi$, $s_0$ and intervention settings) over two years with the daily step used throughout, and requires the relative error to stay below $10^{-12}$ at every step. All states must also stay non-negative.

*Provenance: written with an AI coding agent.*

In [ ]:
# 4.1 Conservation: S + E + I + R = N at every RK4 step, for many random parameter sets
check_rng = np.random.default_rng([SEED, 41])        # independent stream for this check
n_sets = 200
random_params = Params(
    N=N_COLOMBO,
    beta0=check_rng.uniform(0.5, 4.0, n_sets),        # 1/week
    amp=check_rng.uniform(0.0, 0.9, n_sets),
    phase=check_rng.uniform(0, 52, n_sets),           # weeks
    s0=check_rng.uniform(0.1, 1.0, n_sets),
    e0=check_rng.uniform(0, 2000, n_sets),            # people
    i0=check_rng.uniform(1, 2000, n_sets),            # people
    start=check_rng.integers(5, 40, n_sets).astype(float),
    reduction=check_rng.uniform(0.0, 0.8, n_sets),
)
run = simulate(random_params, weeks=104)
S, E, I, R, C = (run["y"][:, k] for k in range(5))  # each has shape (steps + 1, n_sets)

total = S + E + I + R
max_rel_error = np.max(np.abs(total - N_COLOMBO) / N_COLOMBO)
C_expected = (S[0] - S) - (E - E[0])
max_C_error = np.max(np.abs(C - C_expected) / N_COLOMBO)
min_state = run["y"][:, :4].min()

print(f"{n_sets} parameter sets x {run['t'].size} time points")
print(f"max |S+E+I+R - N| / N          = {max_rel_error:.2e}")
print(f"max |C - (people left S and E)| / N = {max_C_error:.2e}")
print(f"smallest value of any compartment = {min_state:.3g} people")

assert max_rel_error < 1e-12, "population is not conserved"
assert max_C_error < 1e-12, "cumulative infections do not match the flows out of S and E"
assert min_state > -1e-6, "a compartment became negative"
print("PASS: population conserved, cumulative infections consistent, all states non-negative")

### 4.2 Limiting case: no outbreak when $R_0 < 1$

If every infectious person causes fewer than one new infection on average, an outbreak cannot grow. In this model an infectious person stays infectious for an exponential time with rate $\gamma$ and infects at rate at most $\beta_{\max} = \beta_0(1+a)$ (when everyone is susceptible), so the expected number of new infections per infectious person is at most

$$R_{0,\max} = \frac{\beta_0\,(1+a)}{\gamma}.$$

When $R_{0,\max} < 1$ this gives two testable consequences, starting from $E_0$ exposed and $I_0$ infectious people in a fully susceptible population ($s_0 = 1$):

1. **No growth.** Weekly new infections decline after the first week (constant $\beta$).
2. **A bound on the final size.** Counting every chain of transmission, the number of new infections can be at most $R_{0,\max}(E_0 + I_0)/(1 - R_{0,\max})$, so the cumulative count satisfies
$$C(t) \le E_0 + \frac{R_{0,\max}\,(E_0 + I_0)}{1 - R_{0,\max}} \quad \text{for all } t,$$
with or without seasonality.

As a control, the same run with $R_0 = 1.5$ must produce a large outbreak, showing the check can tell the two regimes apart.

*Provenance: written with an AI coding agent.*

In [ ]:
# 4.2 Limiting case R0 < 1: infections decline and stay below the branching-process bound
gamma = GAMMA_DEFAULT                     # 1/week
R0_max = 0.9
seed_people = dict(e0=100.0, i0=100.0)    # people
bound = seed_people["e0"] + R0_max * (seed_people["e0"] + seed_people["i0"]) / (1 - R0_max)

print(f"R0,max = {R0_max}; bound on cumulative infections = {bound:,.0f} people")
for amp in [0.0, 0.3, 0.6]:
    p_sub = Params(N=N_COLOMBO, beta0=R0_max * gamma / (1 + amp), amp=amp, phase=22.0, s0=1.0, **seed_people)
    assert np.isclose(basic_reproduction_number(np.linspace(0, 52, 521), p_sub).max(), R0_max, rtol=1e-3)
    out = simulate(p_sub, weeks=52)
    weekly = out["weekly_infections"]
    C_final = out["y"][-1, 4]
    print(f"  amplitude {amp:.1f}: cumulative infections after 52 weeks = {C_final:7,.1f}, "
          f"largest week = {weekly.max():6.1f}")
    assert np.all(out["y"][:, 4] <= bound + 1e-9), "cumulative infections exceed the R0 < 1 bound"
    if amp == 0.0:
        assert np.all(np.diff(weekly[1:]) < 0), "weekly infections grew although R0 < 1"

# Control: the same seeding with R0 = 1.5 does produce an outbreak
p_super = Params(N=N_COLOMBO, beta0=1.5 * gamma, s0=1.0, **seed_people)
weekly_super = simulate(p_super, weeks=52)["weekly_infections"]
print(f"  control R0 = 1.5: {weekly_super.sum():,.0f} infections in 52 weeks (peak week {weekly_super.max():,.0f})")
assert weekly_super.sum() > 100 * bound, "control run should give an outbreak far above the R0 < 1 bound"

fig, ax = plt.subplots()
ax.semilogy(np.arange(1, 53), simulate(Params(N=N_COLOMBO, beta0=R0_max * gamma, s0=1.0, **seed_people), 52)["weekly_infections"],
            label=f"$R_0$ = {R0_max} (no outbreak)")
ax.semilogy(np.arange(1, 53), weekly_super, label="$R_0$ = 1.5 (outbreak)")
ax.set(xlabel="Week", ylabel="New infections per week (log scale)", title="Limiting case: $R_0$ below and above 1")
ax.legend(frameon=False)
plt.show()
print("PASS: with R0 < 1 infections decline and stay below the bound; with R0 > 1 an outbreak occurs")

### 4.3 Convergence order of the RK4 solver

The classical RK4 method has local truncation error $O(h^5)$ per step and, over a fixed time interval, global error $O(h^4)$: halving the step $h$ should cut the error by about $2^4 = 16$. For comparison, the forward Euler method is first order ($O(h)$) and the midpoint RK2 method second order ($O(h^2)$).

**Set-up.** The illustrative scenario from Section 3.2, *including* an intervention that switches on at week 18, is integrated for 45 weeks with steps $h = 1, \tfrac12, \tfrac14, \dots$ weeks. The error is the largest difference over all weeks in the cumulative number of infections $C$ (people) from a **reference solution** computed independently with SciPy's 8th-order Dormand–Prince method (`DOP853`) at relative tolerance $10^{-13}$, restarted at the switch-on time so that it never steps across the jump in $\beta$. SciPy is used only for this check; the model itself is solved with the hand-written RK4.

**Pass criterion.** The slope of $\log(\text{error})$ against $\log h$ for RK4, fitted over $h = \tfrac18$ to $\tfrac1{64}$ week (where the error is well above the reference's own accuracy), must lie between 3.8 and 4.2. Euler and RK2 must show slopes near 1 and 2.

*Provenance: written with an AI coding agent.*

In [ ]:
# 4.3 Error scaling of Euler, RK2 and RK4 against an independent high-accuracy reference
from scipy.integrate import solve_ivp

conv_params = example.with_(start=18, reduction=0.25)   # illustrative scenario with an intervention
P_conv = conv_params.arrays()
CONV_WEEKS = 45


def euler_step(f, t, y, h, *args):
    return y + h * f(t, y, *args)


def rk2_step(f, t, y, h, *args):                          # explicit midpoint method
    k1 = f(t, y, *args)
    return y + h * f(t + h / 2, y + h / 2 * k1, *args)


def integrate(stepper, steps_per_week):
    # Same scheme as simulate(), with any one-step method; returns the state at the end of each week
    h = 1.0 / steps_per_week
    y = initial_state(P_conv)
    weekly = [y]
    for k in range(CONV_WEEKS * steps_per_week):
        t = k * h
        y = stepper(rhs, t, y, h, P_conv, control(t + h / 2, P_conv))
        if (k + 1) % steps_per_week == 0:
            weekly.append(y)
    return np.array(weekly)


# The RK4 path through integrate() must be exactly the model's own solver
assert np.array_equal(integrate(rk4_step, 7), simulate(conv_params, CONV_WEEKS)["y"][::7])

# Reference: DOP853, restarted at the intervention start so it never steps across the jump
y_start = initial_state(P_conv)
reference = [y_start]
for t_a, t_b in [(0, 18), (18, CONV_WEEKS)]:
    u = control((t_a + t_b) / 2, P_conv)
    sol = solve_ivp(lambda t, y: rhs(t, y, P_conv, u), (t_a, t_b), y_start, method="DOP853",
                    rtol=1e-13, atol=1e-9, t_eval=np.arange(t_a + 1, t_b + 1))
    reference.extend(sol.y.T)
    y_start = sol.y[:, -1]
reference = np.array(reference)


def max_error(states):                                  # people, cumulative infections C
    return np.max(np.abs(states[:, 4] - reference[:, 4]))


methods = {"Euler": (euler_step, [8, 16, 32, 64, 128, 256], 1),
           "RK2":   (rk2_step,   [2, 4, 8, 16, 32, 64], 2),
           "RK4":   (rk4_step,   [1, 2, 4, 8, 16, 32, 64], 4)}
results = {}
for name, (stepper, spws, order) in methods.items():
    h = 1.0 / np.array(spws)
    err = np.array([max_error(integrate(stepper, s)) for s in spws])
    fit = h <= 1 / 8 if name == "RK4" else np.ones_like(h, dtype=bool)
    slope = np.polyfit(np.log(h[fit]), np.log(err[fit]), 1)[0]
    results[name] = (h, err, slope, order)
    ratios = err[:-1] / err[1:]
    print(f"{name:5s} fitted order {slope:4.2f} (theory {order}); error ratio per halving: "
          + ", ".join(f"{r:.1f}" for r in ratios))

slope_rk4 = results["RK4"][2]
assert 3.8 <= slope_rk4 <= 4.2, f"RK4 order {slope_rk4:.2f} is outside [3.8, 4.2]"
assert 0.9 <= results["Euler"][2] <= 1.1 and 1.9 <= results["RK2"][2] <= 2.1

default_err = max_error(integrate(rk4_step, 7))
print(f"\nRK4 error at the default step h = 1/7 week (1 day): {default_err:.1e} people "
      f"of {reference[-1, 4]:,.0f} cumulative infections")

fig, ax = plt.subplots(figsize=(7, 5))
for (name, (h, err, slope, order)), marker in zip(results.items(), "osD"):
    ax.loglog(h, err, marker + "-", label=f"{name}: fitted order {slope:.2f}")
    ax.loglog(h, err[-1] * (h / h[-1]) ** order, ":", color="0.5")
ax.axvline(1 / 7, color="C3", lw=0.8, ls="--", label="default step (1 day)")
ax.set(xlabel="Step size h (weeks)", ylabel="Max error in cumulative infections (people)",
       title="Convergence order: dotted guides have slopes 1, 2 and 4")
ax.legend(frameon=False)
fig.savefig(FIG_DIR / "rk4_convergence.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"PASS: RK4 converges at order {slope_rk4:.2f} (expected 4); Euler and RK2 at orders 1 and 2")

The error falls by a factor close to 16 each time the step is halved, as fourth-order theory predicts. At large steps the observed order is above 4 because the leading $h^4$ term does not yet dominate (pre-asymptotic range), which is why the slope is fitted on the smaller steps. The intervention switch does not reduce the order, because the step grid is aligned with the switch-on time and $u(t)$ is held constant within each step. At the one-day step used throughout this notebook the error is a tiny fraction of one person, so the solver error is negligible next to the uncertainty in the data and parameters.

### 4.4 Recovery of known parameters by MCMC

*To be added with the calibration code: the sampler is run on synthetic data generated from known parameters, and each true value must lie inside its 95% credible interval.*

## 5. Experiments and results

*To be added: cases prevented for a grid of intervention start weeks and strengths, computed over posterior samples and reported as medians with 95% credible intervals, followed by a sensitivity analysis of the key inputs.*

## 6. Discussion and limitations

*To be completed with the results.* The main limitations already known are the host-only structure (no explicit mosquito population), a single sinusoidal seasonal pattern, homogeneous mixing across the whole district, and suspected (not laboratory-confirmed) case counts with a constant reporting rate.

**Web demo reference value:** *to be stated once the calibrated results are available.*

## 7. References

1. National Dengue Control Unit, Ministry of Health, Sri Lanka, "Weekly Dengue Update," 2026, weeks 1–45. [Online]. Available: https://www.dengue.health.gov.lk/weekly-report/
2. Department of Census and Statistics, Sri Lanka, "Census of Population and Housing 2024," final report, 2026. [Online]. Available: https://www.statistics.gov.lk/Blog/Blog_details
3. M. Chan and M. A. Johansson, "The incubation periods of dengue viruses," *PLoS ONE*, vol. 7, no. 11, p. e50972, 2012, doi: 10.1371/journal.pone.0050972.
4. World Health Organization, "Dengue and severe dengue," fact sheet. [Online]. Available: https://www.who.int/news-room/fact-sheets/detail/dengue-and-severe-dengue

In [ ]:
elapsed = time.perf_counter() - NOTEBOOK_START
print(f"Notebook ran in {elapsed:.1f} s")
assert elapsed < 600, "Run all should finish within 10 minutes"